# 🏀 NBA Lakehouse — Notebook 2: Bronze → Silver

## Overview
This notebook implements the **second layer** of the Medallion Architecture: reading raw Delta tables from Bronze, applying data quality transformations, and writing clean, strongly-typed Delta tables to the Silver schema.

## Architecture
```
Bronze (raw Delta)  →  Silver (cleaned Delta)
  All eras, all types     NBA only, 2000+, correct types
```

## Transformations Applied
| Transformation | Description |
|---|---|
| League filter | Keep only `lg == 'NBA'` (removes ABA/BAA historical data) |
| Season filter | Keep only `season >= 2000` (modern era) |
| NA handling | Replace string `'NA'` values with `null` |
| Type casting | Cast numeric string columns to `double` or `integer` |
| Null dropping | Drop rows missing critical columns (player, season, key stats) |
| Feature engineering | Add `win_pct` column to team_summaries |

## Design Decisions
- **`'NA'` strings** are common in basketball-reference exports — must be replaced with `null` before casting, otherwise Spark raises `CAST_INVALID_INPUT`
- **Helper functions** `clean_na()` and `cast_numeric()` are defined once and reused across all tables for DRY code
- **Silver does NOT drop all nulls** — only rows missing business-critical fields are dropped; optional stats can remain null

## Step 1 — Imports and Helper Functions

Two reusable helper functions are defined:
- `clean_na(df)` — replaces `'NA'` strings with `null` across all string columns
- `cast_numeric(df, exclude_cols)` — casts all string columns (except excluded ones) to `double`

In [0]:
from pyspark.sql.functions import col, when, round as spark_round, to_date

def clean_na(df):
    """
    Replace string 'NA' values with null across all string columns.
    Basketball-reference exports use 'NA' for missing values instead of
    empty strings or nulls, which causes type casting to fail.
    """
    for c in df.columns:
        if dict(df.dtypes)[c] == "string":
            # Only replace exact 'NA' strings, not partial matches
            df = df.withColumn(c, when(col(c) == "NA", None).otherwise(col(c)))
    return df

def cast_numeric(df, exclude_cols):
    """
    Cast all remaining string columns to double, excluding specified columns.
    Used for tables where most columns are numeric but a few are identifiers
    (team name, player name, etc.) that must stay as strings.
    """
    for c in df.columns:
        # Skip columns that should remain as strings (names, IDs, etc.)
        if c not in exclude_cols and dict(df.dtypes)[c] == "string":
            df = df.withColumn(c, col(c).cast("double"))
    return df

print("✓ Helper functions ready!")

## Step 2 — Silver: player_per_game

The main player stats table. Each row = one player's stats for one season with one team.

**Columns cast to numeric:**
- `age` → integer (player's age during that season)
- `gs` → integer (games started)
- All shooting/stat percentages → double (FG%, 3P%, FT%, etc.)
- All per-game counting stats → double (ORB, DRB, TRB, STL, BLK, TOV, PF)

**Rows dropped:** Players missing `player`, `season`, or `pts_per_game`

In [0]:
# Read from Bronze layer
df = spark.read.table("nba_lakehouse_catalog.bronze.player_per_game")

# Filter to NBA only (excludes historical ABA/BAA leagues) and modern era
df = df.filter((col("lg") == "NBA") & (col("season") >= 2000))

# Replace all 'NA' string values with null before type casting
df = clean_na(df)

# Cast individual columns to their correct types
# age and gs are whole numbers (integer), all stats are decimal (double)
df = df \
    .withColumn("age", col("age").cast("integer")) \
    .withColumn("gs", col("gs").cast("integer")) \
    .withColumn("mp_per_game", col("mp_per_game").cast("double")) \
    .withColumn("fg_percent", col("fg_percent").cast("double")) \
    .withColumn("x3p_per_game", col("x3p_per_game").cast("double")) \
    .withColumn("x3pa_per_game", col("x3pa_per_game").cast("double")) \
    .withColumn("x3p_percent", col("x3p_percent").cast("double")) \
    .withColumn("x2p_per_game", col("x2p_per_game").cast("double")) \
    .withColumn("x2pa_per_game", col("x2pa_per_game").cast("double")) \
    .withColumn("x2p_percent", col("x2p_percent").cast("double")) \
    .withColumn("e_fg_percent", col("e_fg_percent").cast("double")) \
    .withColumn("ft_percent", col("ft_percent").cast("double")) \
    .withColumn("orb_per_game", col("orb_per_game").cast("double")) \
    .withColumn("drb_per_game", col("drb_per_game").cast("double")) \
    .withColumn("trb_per_game", col("trb_per_game").cast("double")) \
    .withColumn("stl_per_game", col("stl_per_game").cast("double")) \
    .withColumn("blk_per_game", col("blk_per_game").cast("double")) \
    .withColumn("tov_per_game", col("tov_per_game").cast("double")) \
    .withColumn("pf_per_game", col("pf_per_game").cast("double"))

# Drop rows missing critical fields — these rows cannot be used for analysis
# The 'lg' column is dropped as all remaining rows are NBA
df = df.dropna(subset=["player", "season", "pts_per_game"]).drop("lg")

# Write to Silver as managed Delta table
df.write.format("delta").mode("overwrite").saveAsTable("nba_lakehouse_catalog.silver.player_per_game")
print(f"✓ silver.player_per_game — {df.count()} rows written")

## Step 3 — Silver: team_summaries

Team season summary table including wins, losses, offensive/defensive ratings, and attendance.

**Feature engineering:** A new `win_pct` column is derived as `w / (w + l)`, rounded to 3 decimal places. This is a key metric for the Gold layer team performance analysis.

In [0]:
# Read team summaries from Bronze
df = spark.read.table("nba_lakehouse_catalog.bronze.team_summaries")

# Filter to NBA and modern era (2000+)
df = df.filter((col("lg") == "NBA") & (col("season") >= 2000))

# Replace 'NA' strings with null
df = clean_na(df)

# Cast numeric columns — wins/losses are integers, ratings are decimals
df = df \
    .withColumn("age", col("age").cast("double")) \
    .withColumn("w", col("w").cast("integer")) \
    .withColumn("l", col("l").cast("integer")) \
    .withColumn("pw", col("pw").cast("double")) \
    .withColumn("pl", col("pl").cast("double")) \
    .withColumn("mov", col("mov").cast("double")) \
    .withColumn("sos", col("sos").cast("double")) \
    .withColumn("srs", col("srs").cast("double")) \
    .withColumn("o_rtg", col("o_rtg").cast("double")) \
    .withColumn("d_rtg", col("d_rtg").cast("double")) \
    .withColumn("n_rtg", col("n_rtg").cast("double")) \
    .withColumn("pace", col("pace").cast("double")) \
    .withColumn("ts_percent", col("ts_percent").cast("double")) \
    .withColumn("attend", col("attend").cast("integer")) \
    .withColumn("attend_g", col("attend_g").cast("integer"))

# Feature engineering: derive win percentage from wins and losses
# Rounded to 3 decimal places for readability (e.g. 0.756)
df = df.withColumn("win_pct", spark_round(col("w") / (col("w") + col("l")), 3))

# Drop rows missing team identity or key performance metrics
df = df.dropna(subset=["team", "season", "w", "o_rtg", "d_rtg"]).drop("lg")

df.write.format("delta").mode("overwrite").saveAsTable("nba_lakehouse_catalog.silver.team_summaries")
print(f"✓ silver.team_summaries — {df.count()} rows written")

## Step 4 — Silver: advanced

Advanced analytics table containing metrics like PER (Player Efficiency Rating), Win Shares, Box Plus/Minus, and VORP (Value Over Replacement Player). These are computed metrics derived from box score data.

**Note:** Only specific known numeric columns are explicitly cast here (rather than using `cast_numeric()`) to avoid accidentally casting any ambiguous columns.

In [0]:
# Read advanced stats from Bronze
df = spark.read.table("nba_lakehouse_catalog.bronze.advanced")

# Filter to NBA and modern era
df = df.filter((col("lg") == "NBA") & (col("season") >= 2000))
df = clean_na(df)

# List of known numeric columns in the advanced stats table
# These are all advanced metrics that should be stored as doubles
numeric_cols = [
    "age",          # Player age
    "per",          # Player Efficiency Rating
    "ts_percent",   # True Shooting %
    "x3p_ar",       # 3-Point Attempt Rate
    "f_tr",         # Free Throw Rate
    "orb_percent",  # Offensive Rebound %
    "drb_percent",  # Defensive Rebound %
    "trb_percent",  # Total Rebound %
    "ast_percent",  # Assist %
    "stl_percent",  # Steal %
    "blk_percent",  # Block %
    "tov_percent",  # Turnover %
    "usg_percent",  # Usage Rate %
    "ows",          # Offensive Win Shares
    "dws",          # Defensive Win Shares
    "ws",           # Total Win Shares
    "ws_per_48",    # Win Shares per 48 minutes (note: stored as ws_48 in some versions)
    "obpm",         # Offensive Box Plus/Minus
    "dbpm",         # Defensive Box Plus/Minus
    "bpm",          # Box Plus/Minus
    "vorp"          # Value Over Replacement Player
]

# Cast only columns that exist in the DataFrame (defensive check)
for c in numeric_cols:
    if c in df.columns:
        df = df.withColumn(c, col(c).cast("double"))

# Drop rows missing player identity
df = df.dropna(subset=["player", "season"]).drop("lg")

df.write.format("delta").mode("overwrite").saveAsTable("nba_lakehouse_catalog.silver.advanced")
print(f"✓ silver.advanced — {df.count()} rows written")

## Step 5 — Silver: team_stats_per_game

Team-level per-game statistics (points, rebounds, assists per game at team level). Uses `cast_numeric()` helper since most columns are numeric — only team name, abbreviation, and season are excluded.

In [0]:
# Read team per-game stats from Bronze
df = spark.read.table("nba_lakehouse_catalog.bronze.team_stats_per_game")
df = df.filter((col("lg") == "NBA") & (col("season") >= 2000))
df = clean_na(df)

# Cast all string columns to double EXCEPT identifiers
# 'team' and 'abbreviation' must stay as strings (team names like 'LAL', 'BOS')
# 'season' is already integer from inferSchema
df = cast_numeric(df, exclude_cols=["lg", "team", "abbreviation", "season"])

df = df.dropna(subset=["team", "season"]).drop("lg")

df.write.format("delta").mode("overwrite").saveAsTable("nba_lakehouse_catalog.silver.team_stats_per_game")
print(f"✓ silver.team_stats_per_game — {df.count()} rows written")

## Step 6 — Silver: player_play_by_play

Play-by-play derived stats per player per season, including on/off court ratings, position-specific metrics, and usage in various game situations.

In [0]:
# Read play-by-play stats from Bronze
df = spark.read.table("nba_lakehouse_catalog.bronze.player_play_by_play")
df = df.filter((col("lg") == "NBA") & (col("season") >= 2000))
df = clean_na(df)

# Exclude all player identity and categorical columns from numeric casting
# 'pos' (position like G, F, C) must stay as string
df = cast_numeric(df, exclude_cols=["lg", "player", "player_id", "team", "pos", "season"])

df = df.dropna(subset=["player", "season"]).drop("lg")

df.write.format("delta").mode("overwrite").saveAsTable("nba_lakehouse_catalog.silver.player_play_by_play")
print(f"✓ silver.player_play_by_play — {df.count()} rows written")

## Step 7 — Silver: player_career_info

Player biographical and career metadata. Unlike other tables, this is **not filtered by season or league** as it contains one row per player across their entire career.

**Type conversions:**
- `ht_in_in` (height in inches) → integer
- `wt` (weight in lbs) → integer  
- `birth_date` and `debut` → date type (parsed from ISO string format)

In [0]:
# Read player career info from Bronze
# Note: No season/league filter — this is a player-level lookup table
df = spark.read.table("nba_lakehouse_catalog.bronze.player_career_info")
df = clean_na(df)

# Cast physical attributes to integers (whole numbers)
df = df \
    .withColumn("ht_in_in", col("ht_in_in").cast("integer")) \
    .withColumn("wt", col("wt").cast("integer"))

# Parse date strings to proper date type
# birth_date format: '1985-02-17'
# debut format: '2003-10-29T00:00:00Z' (ISO 8601 — to_date handles this)
df = df \
    .withColumn("birth_date", to_date(col("birth_date"))) \
    .withColumn("debut", to_date(col("debut")))

# Drop rows missing player identity (cannot be joined to other tables)
df = df.dropna(subset=["player", "player_id"])

df.write.format("delta").mode("overwrite").saveAsTable("nba_lakehouse_catalog.silver.player_career_info")
print(f"✓ silver.player_career_info — {df.count()} rows written")

## Step 8 — Verify Silver Layer

Final verification that all 6 Silver tables were written correctly with expected row and column counts.

In [0]:
# Read back each Silver table and print its dimensions as a sanity check
print("=== Silver Layer Summary ===\n")

silver_tables = [
    "player_per_game",
    "advanced",
    "team_summaries",
    "team_stats_per_game",
    "player_play_by_play",
    "player_career_info"
]

for t in silver_tables:
    df = spark.read.table(f"nba_lakehouse_catalog.silver.{t}")
    print(f"✓ {t}: {df.count()} rows | {len(df.columns)} cols")

## Summary

✅ **Silver layer complete** — 6 clean Delta tables registered in `nba_lakehouse_catalog.silver`

| Table | Bronze Rows | Silver Rows | Reduction |
|-------|-------------|-------------|----------|
| player_per_game | 33,278 | 16,565 | ~50% (NBA + 2000+ filter) |
| advanced | 33,278 | 16,565 | ~50% |
| team_summaries | 1,907 | 805 | ~58% |
| team_stats_per_game | 1,907 | 832 | ~56% |
| player_play_by_play | 18,193 | 16,565 | ~9% |
| player_career_info | 5,396 | 5,396 | 0% (no filter) |

**Next:** Run `03_Silver_To_Gold` to build analytics-ready aggregated tables.